# 03 – Load Data into MySQL
Loads the cleaned CSVs from `data/processed/` into the `nhs_ae` MySQL database
(tables created by `sql/01_create_tables.sql`).

In [1]:
import pandas as pd
from pathlib import Path
from getpass import getpass
from sqlalchemy import create_engine, URL, text

PROCESSED = Path("../data/processed")

password = getpass("MySQL password: ")

url = URL.create("mysql+pymysql", username="root", password=password,
                 host="localhost", port=3306, database="nhs_ae")
engine = create_engine(url)

with engine.connect() as conn:
    print(conn.execute(text("SHOW TABLES")).fetchall())

[('ae_when',), ('ar_activity',)]


## Load cleaned data
Read the cleaned CSVs and append them into the existing MySQL tables. Tables are emptied
first so the notebook can be rerun safely without creating duplicate rows.
**Check:** row counts in MySQL must match the CSVs (7,211 and 640,703).

In [2]:
activity = pd.read_csv(PROCESSED / "ae_activity_clean.csv", parse_dates=["Month"])
when = pd.read_csv(PROCESSED / "ae_when_clean.csv", parse_dates=["Month"])

with engine.begin() as conn:
    conn.execute(text("DELETE FROM ae_activity"))
    conn.execute(text("DELETE FROM ae_when"))

activity.to_sql("ae_activity", engine, if_exists="append", index=False, chunksize=5000)
when.to_sql("ae_when", engine, if_exists="append", index=False, chunksize=5000)

with engine.connect() as conn:
    for table in ["ae_activity", "ae_when"]:
        print(table, conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar())

ae_activity 7211
ae_when 640703
